# C01 — native land: cessions (Royce), state cessions, pre-1784 lines, reserves, removal routes
Task `tasks/T-007-c01-native-layers.md` (brief B-018). Layers 2 and 5 (`docs/PLAN.md` §2). File reads + joins only; no ML/LLM.

**Sources**
| Input | What | Grain | Licence |
|---|---|---|---|
| `data/raw/gis/usfs-tribal-ceded-lands/shp/` + `table/…gdb` | USFS Tribal Ceded Lands = Royce (1899) cession areas, EPSG:4269 | one row per Royce cession number (`CESSNUM`) | CC BY 4.0 |
| `research/treaties_R1.csv` | R1 treaty list (a selection) | treaty | ours |
| `data/processed/reservations_points.csv` | R4 reserves + T-002 points | reserve | ours (coords: OSM/ODbL, Newberry, Royce) |
| `data/raw/gis/nps-trail-of-tears/` | NPS Trail of Tears NHT designated route, 3 GeoJSON pages | line segment | US federal work |
| `data/raw/gis/newberry-ahcb-counties/` | modern county polygons (`END_DATE` 2000), only to place the NY state-cession points | county | free, cite |
| `research/pre1784_lines_R11.csv` | R11 vertices of seven pre-1784 lines, each with `method` and `uncertainty_km` (T-022) | line vertex | ours |
| Natural Earth 10m rivers (read remotely) | river courses between described vertices on a named river (T-022) | river | public domain |
| `web/data/borders.topojson` (E01) | 1784 states and territories, to keep the zone inside the US of 1784 | state version | Newberry AHCB, cite |

**Outputs** (`web/data/`): `native_land.topojson` (objects `royce_cessions`, `state_cessions`, `pre1784_lines`, `before_1784`), `reservations.json`, `removal_routes.topojson` (object `trail_of_tears`).

**Caveats** (R7, `docs/DATA.md`)
- Royce maps the United States' acquisitions, drawn by Royce's staff in the 1890s from treaty wording; it is not a map of where nations lived. Nation names are as the cession schedule writes them (`SCHDTRB`), often decades after 1784.
- Royce has no NY state cessions 1785–89 and no New England areas; those appear here as points (`source = not_royce`) or not at all.
- **Pre-1784 lines (T-022):** hand-traced by the historian in R11 from treaty and deed texts (no open vector exists, R6). One feature per segment between two R11 vertices; a segment takes the weaker `method` and the larger `uncertainty_km` of its ends (floor 3 km). Most vertices are not yet GNIS-checked.
- The Trail of Tears is one NPS designated route, undated, digitised at 1:100,000; it does not say which nation or which year.
- **Shared extent (T-011):** Royce polygons and the Trail of Tears are clipped to the map extent (`extent` in `relief_bounds.json`, written by E01; west edge −100°). Cessions wholly outside drop out; cut ones carry `clipped_to_extent = true`, and their `SHAPE_AREA` / `SHAPE_LEN` stay the USFS values for the whole area.
- **Removal destinations (T-021, R10 §2):** Royce polygons in Indian Territory, Kansas and Iowa whose first USFS date is the treaty that *assigned* them to a removed nation. They carry `assigned_year` / `nation_assigned` / `assigned_verified` / `assigned_treaty`; `cession_year` is the next USFS cession after that (often after 1840), the USFS first year stays in `cession_year_usfs`.
- **Before 1784 (T-021, T-022, R10 §3):** "Taken before 1784: not mapped by Royce" = land east of the R11 zone edge (Fort Stanwix, Donelson, Lochaber, Hard Labour, Augusta 1773 and 1763), inside the US of 1784, minus every Royce polygon. Split into pieces with `uncertainty` (low/medium/high) and a `note`. Three closures are assumed (north of Wood Creek, south of Santa Savilla, augusta_1773 seq 4). Cessions of 1775–83 (Henderson, DeWitt's Corner, Long Island of Holston, Augusta 1783) are not yet drawn.


In [1]:
import sys, json, re
from pathlib import Path
PIPELINE_DIR = Path.cwd().parent
if str(PIPELINE_DIR) not in sys.path: sys.path.insert(0, str(PIPELINE_DIR))
import config
import duckdb, pandas as pd, numpy as np, geopandas as gpd, pyogrio, shapely
from shapely.geometry import shape, mapping
pd.set_option('display.width',220); pd.set_option('display.max_rows',120); pd.set_option('display.max_colwidth',90); pd.set_option('display.max_columns',40)
con = duckdb.connect()
def q(sql): return con.sql(sql).df()

GIS = config.RAW_DIR / 'gis'
ROYCE_SHP = GIS / 'usfs-tribal-ceded-lands/shp/S_USA.TRIBALCEDEDLANDS.shp'
ROYCE_TBL = GIS / 'usfs-tribal-ceded-lands/table/S_USA.TribalCededLandsTable.gdb'
COUNTIES  = GIS / 'newberry-ahcb-counties/shp/US_HistCounties_Gen001.shp'
TRAIL     = sorted((GIS / 'nps-trail-of-tears').glob('TRTE_NHT_off*.geojson'))
R1_CSV    = config.PROJECT_DIR / 'research/treaties_R1.csv'
RES_CSV   = config.PROCESSED_DIR / 'reservations_points.csv'
WEB_DIR   = config.PROJECT_DIR / 'web/data'
WEB_DIR.mkdir(parents=True, exist_ok=True)
WINDOW_END = 9999          # no cutoff: every dated Royce row. L2 = land not yet ceded at the slider date, so areas ceded after 1840 must be drawn (Lara, 2026-11-23)
QUANT = 100_000            # TopoJSON quantization (about 50 m at this extent)
EXTENT = json.loads((WEB_DIR / 'relief_bounds.json').read_text())['extent']   # T-011: shared map extent [west, south, east, north], from E01
EXT_BOX = shapely.box(*EXTENT)

DATASET = {
    'royce': 'USDA Forest Service, Tribal Ceded Lands (Royce 1899), CC BY 4.0',
    'r1':    'R1 treaties (research/treaties_R1.csv)',
    'r4':    'R4 reservations + T-002 points (data/processed/reservations_points.csv)',
    'nps':   'National Park Service 2021, Trail of Tears NHT designated route',
}
print('duckdb', duckdb.__version__, '| geopandas', gpd.__version__, '| shapely', shapely.__version__)

duckdb 2.0.0 | geopandas 1.2.0 | shapely 2.2.0


## 0. TopoJSON writer
No TopoJSON package is installed and the task allows no new dependency, so a small encoder: one arc per ring or line (no shared arcs),
quantized and delta-encoded per the TopoJSON spec. Null geometries (`"type": null`) carry rows with no shape. Polygon rings and the parts of a MultiLineString are each a list of arcs (`[[a]]`), as the spec requires. The decoder is used only to check the round trip in §6;
§6 also decodes every file with `topojson-client` (the web app's decoder, via `node`) as the reference check (T-010).

In [2]:
def _clean(v):
    """JSON-safe scalar: NaN/NaT -> None, numpy -> python, timestamps -> ISO date."""
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT: return None
    if isinstance(v, pd.Timestamp): return v.strftime('%Y-%m-%d')
    if isinstance(v, np.integer): return int(v)
    if isinstance(v, np.floating): return None if np.isnan(v) else float(v)
    if isinstance(v, np.bool_): return bool(v)
    try:
        if pd.isna(v): return None
    except (TypeError, ValueError): pass
    return v

def clean_props(d): return {k: _clean(v) for k, v in d.items()}

def topo_encode(objects, n=QUANT):
    """objects: {name: [(shapely geom or None, props dict, id), ...]} -> TopoJSON dict."""
    geoms = [g for feats in objects.values() for g, _, _ in feats if g is not None]
    x0, y0, x1, y1 = shapely.total_bounds(np.array(geoms, dtype=object))
    kx, ky = (x1 - x0) / (n - 1), (y1 - y0) / (n - 1)
    arcs = []
    def qpt(x, y): return [int(round((x - x0) / kx)), int(round((y - y0) / ky))]
    def arc(coords, ring=False):
        pts = [qpt(x, y) for x, y, *_ in coords]
        dd = [pts[0]] + [p for a, p in zip(pts, pts[1:]) if p != a]
        if len(dd) >= (4 if ring else 2): pts = dd        # keep tiny rings un-deduplicated so none vanish
        delta = [pts[0]] + [[b[0] - a[0], b[1] - a[1]] for a, b in zip(pts, pts[1:])]
        arcs.append(delta); return len(arcs) - 1
    def poly(p): return [[arc(p.exterior.coords, True)]] + [[arc(r.coords, True)] for r in p.interiors]   # a ring is a list of arcs
    def enc(g):
        if g is None: return {'type': None}
        t = g.geom_type
        if t == 'Polygon':         return {'type': t, 'arcs': poly(g)}
        if t == 'MultiPolygon':    return {'type': t, 'arcs': [poly(p) for p in g.geoms]}
        if t == 'LineString':      return {'type': t, 'arcs': [arc(g.coords)]}
        if t == 'MultiLineString': return {'type': t, 'arcs': [[arc(l.coords)] for l in g.geoms]}
        if t == 'Point':           return {'type': t, 'coordinates': qpt(g.x, g.y)}
        raise ValueError(t)
    out = {'type': 'Topology', 'bbox': [x0, y0, x1, y1],
           'transform': {'scale': [kx, ky], 'translate': [x0, y0]}, 'objects': {}, 'arcs': arcs}
    for name, feats in objects.items():
        gs = []
        for g, props, fid in feats:
            e = enc(g); e['properties'] = clean_props(props)
            if fid is not None: e['id'] = fid
            gs.append(e)
        out['objects'][name] = {'type': 'GeometryCollection', 'geometries': gs}
    return out

def topo_decode(topo, name):
    """-> list of (shapely geom or None, props). Inverse of topo_encode (no shared arcs)."""
    (kx, ky), (tx, ty) = topo['transform']['scale'], topo['transform']['translate']
    def arc(i):
        x = y = 0; pts = []
        for dx, dy in topo['arcs'][i]:
            x += dx; y += dy; pts.append((x * kx + tx, y * ky + ty))
        return pts
    def dec(g):
        t = g['type']
        if t is None: return None
        if t == 'Point': x, y = g['coordinates']; return shapely.Point(x * kx + tx, y * ky + ty)
        if t == 'LineString': return shapely.LineString(arc(g['arcs'][0]))
        if t == 'MultiLineString': return shapely.MultiLineString([arc(a[0]) for a in g['arcs']])
        if t == 'Polygon': return shapely.Polygon(arc(g['arcs'][0][0]), [arc(a[0]) for a in g['arcs'][1:]])
        if t == 'MultiPolygon': return shapely.MultiPolygon([(arc(p[0][0]), [arc(a[0]) for a in p[1:]]) for p in g['arcs']])
        raise ValueError(t)
    return [(dec(g), g.get('properties', {})) for g in topo['objects'][name]['geometries']]

def write_json(obj, path):
    s = json.dumps(obj, ensure_ascii=False, allow_nan=False, separators=(',', ':'))
    path.write_text(s, encoding='utf-8'); print(f'wrote {path.relative_to(config.PROJECT_DIR)}  {len(s)/1e6:.2f} MB')

# self-test on a square with a hole, a line and a point
_t = topo_encode({'t': [(shapely.Polygon([(0,0),(0,1),(1,1),(1,0)], [[(.2,.2),(.4,.2),(.4,.4),(.2,.4)]]), {'a': np.nan}, 'x'),
                        (shapely.MultiLineString([[(0,0),(1,1)], [(1,0),(0,1)]]), {}, None),
                        (shapely.Point(.5,.5), {}, None), (None, {'b': 1}, None),
                        (shapely.MultiPolygon([shapely.box(2,2,3,3), shapely.box(4,4,5,5)]), {}, None)]})
_d = topo_decode(_t, 't')
assert _d[0][0].equals_exact(shapely.Polygon([(0,0),(0,1),(1,1),(1,0)], [[(.2,.2),(.4,.2),(.4,.4),(.2,.4)]]), 1e-4)
assert _d[1][0].geom_type == 'MultiLineString' and len(_d[1][0].geoms) == 2 and _d[3][0] is None and _d[0][1]['a'] is None
_g = _t['objects']['t']['geometries']
assert _g[0]['arcs'] == [[0], [1]] and _g[1]['arcs'] == [[2], [3]] and _g[4]['arcs'] == [[[4]], [[5]]], [g.get('arcs') for g in _g]
assert _d[4][0].equals_exact(shapely.MultiPolygon([shapely.box(2,2,3,3), shapely.box(4,4,5,5)]), 1e-4)
print('topojson self-test ok')

TOPO_CHECK_JS = r'''
const tc = require(process.argv[1]), fs = require('fs'), out = {};
const pt = p => Array.isArray(p) && p.length === 2 && Number.isFinite(p[0]) && Number.isFinite(p[1]);
for (const f of process.argv.slice(2)) {
  const t = JSON.parse(fs.readFileSync(f, 'utf8')); out[f] = {};
  for (const [k, o] of Object.entries(t.objects)) {
    const r = {features: 0, null: 0, points: 0, rings: 0, bad_rings: 0, lines: 0, bad_lines: 0, bad_type: 0, x0: Infinity, y0: Infinity, x1: -Infinity, y1: -Infinity};
    const bb = p => { r.x0 = Math.min(r.x0, p[0]); r.y0 = Math.min(r.y0, p[1]); r.x1 = Math.max(r.x1, p[0]); r.y1 = Math.max(r.y1, p[1]); };   // T-011: bbox of every position
    const ring = c => { r.rings++; c.forEach(bb); const ok = c.length >= 4 && c.every(pt) && c[0][0] === c[c.length-1][0] && c[0][1] === c[c.length-1][1]; if (!ok) r.bad_rings++; };
    const line = c => { r.lines++; c.forEach(bb); if (!(c.length >= 2 && c.every(pt))) r.bad_lines++; };
    for (const ft of tc.feature(t, o).features) {
      r.features++; const g = ft.geometry;
      if (!g) { r.null++; continue; }
      if (g.type === 'Polygon') g.coordinates.forEach(ring);
      else if (g.type === 'MultiPolygon') g.coordinates.forEach(p => p.forEach(ring));
      else if (g.type === 'LineString') line(g.coordinates);
      else if (g.type === 'MultiLineString') g.coordinates.forEach(line);
      else if (g.type === 'Point') { r.points += pt(g.coordinates) ? 1 : 0; bb(g.coordinates); }
      else r.bad_type++;
    }
    out[f][k] = r;
  }
}
console.log(JSON.stringify(out));
'''
def topojson_client_check(paths):
    """-> DataFrame (file, object) of counts from topojson-client; raises if node fails."""
    import subprocess
    lib = config.PROJECT_DIR / 'web' / 'node_modules' / 'topojson-client'
    r = subprocess.run(['node', '-e', TOPO_CHECK_JS, str(lib)] + [str(p) for p in paths], capture_output=True, text=True, check=True)
    return pd.DataFrame([dict(file=Path(f).name, object=k, **v) for f, objs in json.loads(r.stdout).items() for k, v in objs.items()])

def assert_inside(chk, extent, tol=1e-9):
    """T-011: every decoded position of every object lies inside extent [west, south, east, north] (objects with no geometry: NaN bbox, skipped)."""
    w, s, e, n = extent; c = chk.dropna(subset=['x0'])
    bad = c[(c.x0 < w - tol) | (c.y0 < s - tol) | (c.x1 > e + tol) | (c.y1 > n + tol)]
    assert bad.empty, bad[['file', 'object', 'x0', 'y0', 'x1', 'y1']]
    return c[['file', 'object', 'x0', 'y0', 'x1', 'y1']]

def clip_to_extent(g, kind):
    """T-011: the part of g inside EXT_BOX, keeping only `kind` parts ('Polygon' or 'LineString'). -> (geometry or None, status)."""
    if g is None or (isinstance(g, float) and np.isnan(g)): return None, 'no_geometry'
    if EXT_BOX.covers(g): return g, 'inside'
    x = shapely.intersection(shapely.make_valid(g), EXT_BOX)
    parts = [p for p in shapely.get_parts(shapely.get_parts(x)) if p.geom_type == kind and not p.is_empty]
    if not parts: return None, 'outside'
    multi = shapely.MultiPolygon if kind == 'Polygon' else shapely.MultiLineString
    return (parts[0] if len(parts) == 1 else multi(parts)), 'clipped'


topojson self-test ok


## 1. Royce cession areas (USFS Tribal Ceded Lands)
The shapefile holds geometry + `CESSNUM`; the table holds the dates (`CESSDATE1..5`), the nations as the schedule names them (`SCHDTRB`),
present-day nations, citations and links. Join on `CESSNUM`. The **table drives**: a dated Royce row with no polygon is kept with a null geometry and `flag = no_geometry`.

In [3]:
shp = gpd.read_file(ROYCE_SHP)
tbl = pyogrio.read_dataframe(ROYCE_TBL)
DATE_COLS = [f'CESSDATE{i}' for i in range(1, 6)]
tbl['_earliest'] = tbl[DATE_COLS].min(axis=1)
print('shapefile', len(shp), shp.crs, '| table', len(tbl))
print('null CESSNUM: shp', shp.CESSNUM.isna().sum(), '| table', tbl.CESSNUM.isna().sum(), '| duplicate CESSNUM: shp', shp.CESSNUM.dropna().duplicated().sum(), '| table', tbl.CESSNUM.dropna().duplicated().sum())
print('table rows with no date:', tbl._earliest.isna().sum(), '| earliest', tbl._earliest.min().date(), '| latest', tbl._earliest.max().date())
print('table CESSNUMs with no polygon:', sorted(set(tbl.CESSNUM.dropna()) - set(shp.CESSNUM.dropna())))
print('polygons with no table row:', sorted(set(shp.CESSNUM.dropna()) - set(tbl.CESSNUM.dropna())), '| shp row with null CESSNUM bounds:', shp[shp.CESSNUM.isna()].total_bounds.round(1))
print('dates by stored hour (UTC):', tbl[DATE_COLS].stack().dt.hour.value_counts().to_dict(), '-> read as calendar dates (see 1a for a one-day offset)')
print('rows by number of dates:', tbl[DATE_COLS].notna().sum(axis=1).value_counts().sort_index().to_dict())

shapefile 718 EPSG:4269 | table 725
null CESSNUM: shp 1 | table 1 | duplicate CESSNUM: shp 0 | table 0
table rows with no date: 1 | earliest 1784-10-21 | latest 1893-02-19
table CESSNUMs with no polygon: ['128', '129', '130', '47', '543', '65', '690']
polygons with no table row: [] | shp row with null CESSNUM bounds: [-110.9   32.5 -109.1   33.4]
dates by stored hour (UTC): {12.0: 1166, 0.0: 4} -> read as calendar dates (see 1a for a one-day offset)
rows by number of dates: {0: 1, 1: 405, 2: 221, 3: 72, 4: 23, 5: 3}


In [4]:
# Scope: every dated Royce row (areas ceded after 1840 stay 'not yet ceded' for the whole 1784–1840 slider)
roy = tbl[tbl._earliest.dt.year <= WINDOW_END].copy()
roy = roy.merge(shp.to_crs(4326)[['CESSNUM', 'CESSNUMSOR', 'SHAPE_AREA', 'SHAPE_LEN', 'geometry']], on='CESSNUM', how='left', validate='1:1')
roy['cessnum'] = roy.CESSNUM
roy['nation_as_named'] = roy.SCHDTRB
roy['cession_date'] = roy._earliest.dt.strftime('%Y-%m-%d')
roy['cession_year'] = roy._earliest.dt.year.astype(int)
roy['cession_dates'] = roy[DATE_COLS].apply(lambda r: ';'.join(sorted(d.strftime('%Y-%m-%d') for d in r.dropna())), axis=1)
roy['n_dates'] = roy[DATE_COLS].notna().sum(axis=1)
for c in DATE_COLS: roy[c] = roy[c].dt.strftime('%Y-%m-%d')
roy['has_geometry'] = roy.geometry.notna()
roy['flag'] = np.where(roy.has_geometry, None, 'no_geometry')
roy['kind'] = 'cession_polygon'; roy['source'] = 'royce'; roy['dataset'] = DATASET['royce']
roy['year'] = roy.cession_year; roy['year_flag'] = None; roy['todo'] = False

# R1 labels: R1 rows that list this Royce number
r1 = pd.read_csv(R1_CSV, dtype=str)
r1x = r1.assign(cessnum=r1.royce_cession_numbers.str.split(';')).explode('cessnum').dropna(subset=['cessnum'])
r1x['cessnum'] = r1x.cessnum.str.strip()
lab = r1x.groupby('cessnum').agg(r1_treaty_id=('id', ';'.join), r1_treaty_name=('name', ';'.join), r1_date_signed=('date_signed', ';'.join)).reset_index()
roy = roy.merge(lab, on='cessnum', how='left', validate='1:1').drop(columns='_earliest')

print('Royce rows (all dated):', len(roy), '| with polygon:', roy.has_geometry.sum(), '| no polygon:', (~roy.has_geometry).sum())
print('invalid polygons in scope:', (~roy[roy.has_geometry].geometry.apply(lambda g: g.is_valid)).sum())
print('with an R1 label:', roy.r1_treaty_id.notna().sum(), '| R1 Royce numbers not in scope:', sorted(set(r1x.cessnum) - set(roy.cessnum)))
roy.loc[~roy.has_geometry, ['cessnum', 'MAPNAME', 'SCHDTRB', 'cession_date', 'STATE']]
print('ceded after 1840:', (roy.cession_year > 1840).sum(), '| earliest date on 31 Dec:', (roy.cession_date.str.endswith('-12-31')).sum())


Royce rows (all dated): 724 | with polygon: 717 | no polygon: 7
invalid polygons in scope: 3
with an R1 label: 65 | R1 Royce numbers not in scope: []
ceded after 1840: 425 | earliest date on 31 Dec: 2


### 1b. Clip to the shared map extent (T-011)
Polygons crossing the extent are cut to it (`clipped_to_extent = true`); polygons wholly outside it drop out. Rows with no polygon stay (they have nothing to clip).

In [5]:
roy_n0 = len(roy)
cut = [clip_to_extent(g, 'Polygon') for g in roy.geometry]
roy['_clip'] = [s for _, s in cut]
roy['geometry'] = [g for g, _ in cut]
roy['clipped_to_extent'] = roy._clip == 'clipped'
roy_out = roy[roy._clip == 'outside']
display(roy._clip.value_counts().to_frame('royce_rows'))
print('dropped (wholly outside), by STATE:', roy_out.STATE.value_counts().to_dict())
print('dropped by cession year: <= 1840', int((roy_out.cession_year <= 1840).sum()), '| > 1840', int((roy_out.cession_year > 1840).sum()),
      '| cut, <= 1840:', int(((roy._clip == 'clipped') & (roy.cession_year <= 1840)).sum()))
roy = roy[roy._clip != 'outside'].drop(columns='_clip').reset_index(drop=True)
print(f'royce_cessions: {roy_n0} rows before -> {len(roy)} after ({roy_n0 - len(roy)} dropped) | polygons {int(roy.has_geometry.sum())} | cut at the edge {int(roy.clipped_to_extent.sum())}')

,royce_rows
_clip,
inside,405
outside,274
clipped,38
no_geometry,7


dropped (wholly outside), by STATE: {'CA': 52, 'WA': 36, 'NM': 35, 'AZ': 31, 'OR': 22, 'MT': 17, 'MN': 11, 'ND': 10, 'UT': 7, 'NV': 7, 'ID': 6, 'CO': 6, 'SD': 4, 'NE': 2, 'MT; ND': 2, 'MT; WY': 2, 'OR; WA': 2, 'CA; NV': 2, 'ND; SD': 2, 'CA; OR': 2, 'WY': 2, 'CO; KS; NE; WY': 1, 'MT; ND; WY': 1, 'ID; OR; WA': 1, 'CA; ID; NV; OR; UT': 1, 'ID; NV': 1, 'NV; UT': 1, 'CA; NV; UT': 1, 'NM; TX': 1, 'NM; UT': 1, 'CO; UT': 1, 'CO; NV; UT': 1, 'CO; ID; MT; UT; WY': 1, 'ID; MT; WA': 1, 'ID; MT': 1}
dropped by cession year: <= 1840 0 | > 1840 274 | cut, <= 1840: 8
royce_cessions: 724 rows before -> 450 after (274 dropped) | polygons 443 | cut at the edge 38


### 1a. Overlaps and R1 date check
Overlapping cessions stay separate polygons (one per Royce number); nothing is dissolved. R1 dates are compared with Royce's earliest date for the same number.
This is a check only: the USFS dates are written to the files as stored. Where R1's treaty is the polygon's first cession, the USFS date is one day earlier than R1 (Kappler) in every case;
the four dates stored at 00:00 UTC are not shifted. Years and decades are not affected unless an earliest date falls on 31 December (checked in the print below). The rows below with large gaps are polygons R1 cites for a later, second cession.

In [6]:
g = gpd.GeoDataFrame(roy[roy.has_geometry][['cessnum', 'geometry']], crs=4326).to_crs(5070)
g['geometry'] = g.geometry.make_valid()
pairs = gpd.sjoin(g, g, predicate='intersects')
pairs = pairs[pairs.cessnum_left < pairs.cessnum_right]
right = g.set_index('cessnum').geometry
pairs['shared_km2'] = [a.intersection(right[b]).area / 1e6 for a, b in zip(pairs.geometry, pairs.cessnum_right)]
big = pairs[pairs.shared_km2 > 1]
print('polygon pairs sharing > 1 km2:', len(big), '| polygons involved:', len(set(big.cessnum_left) | set(big.cessnum_right)))

# R1 date_signed vs Royce earliest date, per (R1 treaty, Royce number)
chk = r1x.merge(roy[['cessnum', 'cession_date', 'cession_dates']], on='cessnum')
chk['days_r1_minus_royce'] = (pd.to_datetime(chk.date_signed) - pd.to_datetime(chk.cession_date)).dt.days
print('days R1 minus Royce earliest:', chk.days_r1_minus_royce.value_counts().sort_index().to_dict())
print('Royce dates stored at 00:00 UTC (the rest are 12:00):')
display(tbl[DATE_COLS].stack().loc[lambda s: s.dt.hour == 0].rename('date').reset_index().merge(tbl[['CESSNUM']], left_on='level_0', right_index=True)[['CESSNUM', 'level_1', 'date']])
print('in-scope Royce earliest dates on 31 Dec (a one-day shift would change the year):', (roy.cession_date.str.endswith('-12-31')).sum())
chk[chk.days_r1_minus_royce != 1][['id', 'cessnum', 'date_signed', 'cession_date', 'cession_dates', 'days_r1_minus_royce']]

polygon pairs sharing > 1 km2: 423 | polygons involved: 356
days R1 minus Royce earliest: {1: 56, 335: 1, 2200: 1, 3157: 1, 4412: 1, 4874: 4, 9082: 1}
Royce dates stored at 00:00 UTC (the rest are 12:00):


,CESSNUM,level_1,date
0,272,CESSDATE2,1856-02-05 00:00:00+00:00
1,128,CESSDATE2,1826-01-24 00:00:00+00:00
2,129,CESSDATE2,1826-01-24 00:00:00+00:00
3,130,CESSDATE2,1826-01-24 00:00:00+00:00


in-scope Royce earliest dates on 31 Dec (a one-day shift would change the year): 0


,id,cessnum,date_signed,cession_date,cession_dates,days_r1_minus_royce
47,T1819,105,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
49,T1819,107,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
50,T1819,108,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
51,T1819,109,1819-02-27,1805-10-24,1805-10-24;1819-02-26,4874
56,T1830,157,1830-09-27,1805-11-15,1805-11-15;1830-09-26,9082
58,T1832S,173,1832-05-09,1823-09-17,1823-09-17;1832-05-08,3157
61,T1833,188,1833-09-26,1821-08-28,1821-08-28;1832-10-26;1833-09-26,4412
62,T1833,189,1833-09-26,1827-09-18,1827-09-18;1833-09-26,2200
63,T1833,190,1833-09-26,1832-10-26,1832-10-26;1833-09-26,335


In [7]:
con.register('roy_v', roy.drop(columns='geometry'))
q("""select (cession_year::int // 10) * 10 as decade, count(*) royce_rows, count(*) filter (where has_geometry) polygons,
     count(*) filter (where n_dates > 1) multi_date, count(*) filter (where r1_treaty_id is not null) with_r1
     from roy_v group by 1 order by 1""")

,decade,royce_rows,polygons,multi_date,with_r1
0,1780,6,6,3,3
1,1790,36,36,7,23
2,1800,55,52,33,17
3,1810,71,71,45,10
4,1820,47,45,23,6
5,1830,84,84,51,6
6,1840,18,18,10,0
7,1850,75,75,35,0
8,1860,26,26,12,0
9,1870,19,18,9,0


### 1c. Removal destinations: dated as assignments (T-021)
`research/destinations_R10.csv` (R10 addendum) names the Royce polygons that were **assigned** to removed nations in 1825–38. Their first USFS date is that granting treaty, so it is not a cession.
For each row with an `assigned_date`: `cession_year` / `cession_date` = the first USFS date in a later year than `assigned_year` (the USFS dates run one day early, and a same-year second date is a supplement or a second grant, e.g. Castor Hill 26 and 28 Oct 1832).
If USFS has no later date, `cession_year` is null with `year_flag = no_later_usfs_cession` (not ceded in the USFS data). Rows with no `assigned_date` keep the USFS dates and get `todo = true`.
The USFS values stay in `cession_year_usfs` / `cession_date_usfs`. `attr_flag` marks the one attribute R10 found wrong (area 492); it is not changed.

In [8]:
DEST_CSV = config.PROJECT_DIR / 'research/destinations_R10.csv'
dest = pd.read_csv(DEST_CSV, dtype=str)
assert dest.cessnum.notna().all() and not dest.cessnum.duplicated().any() and set(dest.verified) <= {'y', 'n'}
print('destinations_R10:', len(dest), 'rows | verified', dest.verified.value_counts().to_dict(), '| with assigned_date', int(dest.assigned_date.notna().sum()),
      '| not in royce_cessions:', sorted(set(dest.cessnum) - set(roy.cessnum)))
dest = dest.rename(columns={'treaty': 'assigned_treaty', 'kappler_vol_page': 'assigned_kappler', 'verified': 'assigned_verified',
                            'prior_occupants': 'prior_occupants_r10', 'note': 'assigned_note'})
dest['assigned_year'] = pd.to_datetime(dest.assigned_date).dt.year.astype('Int64')
roy = roy.merge(dest, on='cessnum', how='left', validate='1:1')
roy['is_destination'] = roy.cessnum.isin(dest.cessnum)
roy['cession_year_usfs'] = roy.cession_year; roy['cession_date_usfs'] = roy.cession_date
roy['cession_year'] = roy.cession_year.astype('Int64')
roy['todo_note'] = None

def next_cession(dates, ay):
    """First USFS date (YYYY-MM-DD; sorted) in a later year than the assigning treaty, else None."""
    later = [d for d in dates.split(';') if int(d[:4]) > ay]
    return later[0] if later else None

dated = roy.is_destination & roy.assigned_year.notna()
nxt = pd.Series([next_cession(d, int(a)) if m else None for d, a, m in zip(roy.cession_dates, roy.assigned_year, dated)], index=roy.index)
roy['redated'] = np.select([dated & nxt.notna(), dated & nxt.isna(), roy.is_destination], ['next_usfs_cession', 'no_later_usfs_cession', 'todo_no_assigned_date'], None)
roy.loc[dated, 'cession_date'] = nxt[dated]
roy.loc[dated, 'cession_year'] = nxt[dated].str[:4].astype(float).astype('Int64')
roy['year'] = roy.cession_year
roy.loc[roy.redated == 'no_later_usfs_cession', 'year_flag'] = 'no_later_usfs_cession'
tod = roy.redated == 'todo_no_assigned_date'
roy.loc[tod, 'todo'] = True
roy.loc[tod, 'todo_note'] = 'T-021: no assigned_date in destinations_R10.csv; USFS dates kept. Check the Royce schedule (see assigned_note).'

# step 4: flag, don't change
roy['attr_flag'] = None
m492 = roy.cessnum == '492'
assert m492.sum() == 1 and 'Santee Sioux' in roy.loc[m492, 'PRESDAYTRB'].iloc[0] and 'Cherokee' in roy.loc[m492, 'SCHDTRB'].iloc[0]
roy.loc[m492, 'attr_flag'] = 'PRESDAYTRB: USFS present-day nation "Santee Sioux Nation, Nebraska" on a Cherokee polygon; likely an attribute error (R10 §2). Not changed.'

d_ = roy[roy.is_destination]
display(pd.crosstab(d_.redated, d_.assigned_verified, margins=True).rename_axis(index='redated', columns='verified'))
d_[['cessnum', 'nation_assigned', 'assigned_date', 'assigned_verified', 'cession_date_usfs', 'cession_date', 'cession_year', 'cession_dates', 'redated', 'STATE']]

destinations_R10: 42 rows | verified {'y': 23, 'n': 19} | with assigned_date 37 | not in royce_cessions: []


verified,n,y,All
redated,,,
next_usfs_cession,13,22,35
no_later_usfs_cession,1,1,2
todo_no_assigned_date,5,0,5
All,19,23,42


,cessnum,nation_assigned,assigned_date,assigned_verified,cession_date_usfs,cession_date,cession_year,cession_dates,redated,STATE
265,500,Seneca and Shawnee (united tribe),1832-12-29,n,1832-12-28,1867-02-22,1867,1832-12-28;1867-02-22;1867-02-22,next_usfs_cession,OK
266,501,Seneca and Shawnee (united tribe),1832-12-29,n,1832-12-28,1867-02-22,1867,1832-12-28;1867-02-22;1867-02-22,next_usfs_cession,OK
267,498,Seneca and Shawnee (united tribe),1832-12-29,n,1832-12-28,1867-02-22,1867,1832-12-28;1867-02-22;1867-02-22,next_usfs_cession,OK
268,502,NaN,NaN,n,1867-02-22,1867-02-22,1867,1867-02-22,todo_no_assigned_date,OK
269,504,Seneca and Shawnee (united tribe); Quapaw,NaN,n,1832-12-28,1832-12-28,1832,1832-12-28;1833-05-12;1867-02-22;1867-02-22,todo_no_assigned_date,OK
285,144,Cherokee nation west of Mississippi river (Western Cherokee),1828-05-06,n,1827-05-05,NaN,<NA>,1827-05-05,no_later_usfs_cession,OK
287,323,Iowa,1836-09-17,y,1836-09-16,1854-05-16,1854,1836-09-16;1854-05-16;1854-05-17;1861-03-05,next_usfs_cession,KS
288,324,Sac and Fox of the Missouri,1836-09-17,y,1836-09-16,1854-05-17,1854,1836-09-16;1854-05-17,next_usfs_cession,KS
289,325,Kickapoo,1832-10-24,n,1832-11-25,1854-05-17,1854,1832-11-25;1854-05-17,next_usfs_cession,KS
291,328,Kaskaskia; Peoria; Piankashaw; Wea,1832-10-27,y,1832-10-26,1854-05-29,1854,1832-10-26;1832-10-28;1854-05-29;1867-02-22,next_usfs_cession,KS


In [9]:
# 1784–1840 check: destination polygons whose new cession year still falls in the window, or where the USFS first date is before the assigning treaty
d_[(d_.cession_year <= 1840) | (d_.cession_year_usfs < d_.assigned_year)][['cessnum', 'nation_assigned', 'assigned_date', 'assigned_verified', 'cession_year_usfs', 'cession_year', 'cession_dates', 'assigned_note']]

,cessnum,nation_assigned,assigned_date,assigned_verified,cession_year_usfs,cession_year,cession_dates,assigned_note
269,504,Seneca and Shawnee (united tribe); Quapaw,NaN,n,1832,1832,1832-12-28;1833-05-12;1867-02-22;1867-02-22,USFS dates both 1832-12-29 and 1833-05-13. Which grant applies is unknown.
285,144,Cherokee nation west of Mississippi river (Western Cherokee),1828-05-06,n,1827,<NA>,1827-05-05,USFS first date is 1827 (one year early); Royce OCR p.720 says May 1828. Area-to-tract...
293,329,Miami,1838-11-06,n,1838,1840,1838-11-05;1840-11-27;1854-06-04,Kansas. Art. 10 promises a country west of the Mississippi but names no place. USFS da...
294,330,Miami,1838-11-06,n,1838,1840,1838-11-05;1840-11-27;1854-06-04;1873-03-02,As 329.
304,267,Winnebago,1832-09-15,y,1832,1837,1832-09-14;1837-10-31;1846-10-12,Iowa/Minnesota. Art. II: the Neutral Ground; leave the ceded land by 1833-06-01. Year ...
325,418,Chippewa of Swan Creek and Black River,NaN,n,1836,1836,1836-05-08;1859-07-15,As 417.
326,417,Chippewa of Swan Creek and Black River; Christian Indians or Munsee,NaN,n,1836,1836,1836-05-08;1859-07-15,Kansas. USFS first date is 1836-05-09 (Royce p.758); that treaty was not opened. The M...
409,430,Delaware,NaN,n,1836,1836,1836-09-16;1854-05-16;1861-03-05,USFS first date is 1836-09-17 (the Iowa etc. treaty). How it relates to the Delaware i...


### 1d. Prior occupants: Royce polygons ceded before the assignment (T-021, report only)
For each dated destination: every other Royce polygon ceded (its `cession_year`) before `assigned_year` that overlaps it, with the share of the destination's area it covers (EPSG:5070).
`union_share` = share of the destination covered by all of them together. Before `assigned_year` the map shows these older polygons, not the destination. Nothing is changed here.

In [10]:
g5 = gpd.GeoDataFrame(roy[roy.has_geometry][['cessnum', 'nation_as_named', 'cession_year', 'geometry']], crs=4326).to_crs(5070)
g5['geometry'] = g5.geometry.make_valid()
G5 = g5.set_index('cessnum')
rows, summ = [], []
for r in roy[dated & roy.has_geometry].itertuples():
    dg = G5.geometry[r.cessnum]; ay = int(r.assigned_year); a = dg.area
    cand = g5[(g5.cessnum != r.cessnum) & g5.cession_year.notna() & (g5.cession_year < ay) & g5.intersects(dg)]
    parts = []
    for c in cand.itertuples():
        x = dg.intersection(c.geometry)
        if x.area / 1e6 > 1:
            parts.append(x); rows.append(dict(destination=r.cessnum, nation_assigned=r.nation_assigned, assigned_year=ay, prior_cessnum=c.cessnum,
                                              prior_nation=c.nation_as_named, prior_cession_year=int(c.cession_year), share=round(x.area / a, 3)))
    u = shapely.union_all(parts).area / a if parts else 0.0
    summ.append(dict(destination=r.cessnum, nation_assigned=r.nation_assigned, assigned_year=ay, dest_km2=round(a / 1e6), n_prior=len(parts),
                     union_share=round(u, 3), prior_occupants_r10=r.prior_occupants_r10))
prior = pd.DataFrame(rows); prior_sum = pd.DataFrame(summ)
print('dated destinations with a polygon:', len(prior_sum), '| with >= 1 prior polygon:', int((prior_sum.n_prior > 0).sum()),
      '| median union_share', prior_sum.union_share.median(), '| fully covered (>= 0.99):', int((prior_sum.union_share >= .99).sum()))
display(prior_sum)
prior.sort_values(['destination', 'share'], ascending=[True, False])

dated destinations with a polygon: 37 | with >= 1 prior polygon: 34 | median union_share 1.0 | fully covered (>= 0.99): 34


,destination,nation_assigned,assigned_year,dest_km2,n_prior,union_share,prior_occupants_r10
0,500,Seneca and Shawnee (united tribe),1832,140,1,1.0,Great and Little Osage; Quapaw (region; overlap not computed)
1,501,Seneca and Shawnee (united tribe),1832,77,1,1.0,Great and Little Osage; Quapaw (region; overlap not computed)
2,498,Seneca and Shawnee (united tribe),1832,91,1,1.0,Great and Little Osage; Quapaw (region; overlap not computed)
3,144,Cherokee nation west of Mississippi river (Western Cherokee),1828,25,1,1.0,Great and Little Osage; Quapaw (region; overlap not computed)
4,323,Iowa,1836,374,1,1.0,Kansas (region; overlap not computed)
5,324,Sac and Fox of the Missouri,1836,495,1,1.0,Kansas (region; overlap not computed)
6,325,Kickapoo,1832,1948,1,1.0,Kansas (region; overlap not computed)
7,328,Kaskaskia; Peoria; Piankashaw; Wea,1832,44,1,1.0,Great and Little Osage (region; overlap not computed)
8,327,Kaskaskia; Peoria; Piankashaw; Wea,1832,255,2,1.0,Great and Little Osage (region; overlap not computed)
9,329,Miami,1838,1379,1,1.0,Great and Little Osage (region; overlap not computed)


,destination,nation_assigned,assigned_year,prior_cessnum,prior_nation,prior_cession_year,share
3,144,Cherokee nation west of Mississippi river (Western Cherokee),1828,97,Great and Little Osage,1818,1.000
13,248,"New York Indians (Seneca, Onondaga, Cayuga, Tuscarora, Oneida, St. Regis, Stockbridge,...",1838,123,Great and Little Osage,1825,1.000
12,249,"New York Indians (Seneca, Onondaga, Cayuga, Tuscarora, Oneida, St. Regis, Stockbridge,...",1838,123,Great and Little Osage,1825,1.000
15,265,"United Nation of Chippewa, Ottawa and Potawatomi",1833,151,Iowa; Medewakanton; Missouri; Omaha; Oto; Sauk and Fox; Sioux (Wahpeton and Sisseton B...,1830,1.000
17,267,Winnebago,1832,153,Iowa; Medewakanton; Missouri; Omaha; Oto; Sauk and Fox; Sioux (Wahpeton and Sisseton B...,1830,0.526
16,267,Winnebago,1832,152,Iowa; Medewakanton; Missouri; Omaha; Oto; Sauk and Fox; Sioux (Wahpeton and Sisseton B...,1830,0.474
20,316,Delaware,1829,124,Kansas,1825,1.000
23,317,Delaware,1829,124,Kansas,1825,1.000
4,323,Iowa,1836,124,Kansas,1825,1.000
5,324,Sac and Fox of the Missouri,1836,124,Kansas,1825,1.000


In [11]:
con.register('roy_v', roy.drop(columns='geometry'))   # re-register with the T-021 columns (used in §6)
q("""select (cession_year::int // 10) * 10 as decade, count(*) royce_rows, count(*) filter (where is_destination) destinations,
     count(*) filter (where (cession_year_usfs::int // 10) * 10 is distinct from (cession_year::int // 10) * 10) moved_decade
     from roy_v group by 1 order by 1""")

,decade,royce_rows,destinations,moved_decade
0,1780,6,0,0
1,1790,36,0,0
2,1800,55,0,0
3,1810,71,0,0
4,1820,43,0,0
5,1830,56,5,0
6,1840,21,3,3
7,1850,90,19,15
8,1860,37,12,11
9,1870,20,1,1


## 2. NY state and private cessions not in Royce (R1) → dated points
R1 rows with no Royce number, sold to New York State or to Phelps and Gorham, 1785–89. Point = centroid of the modern counties R1's `present_day_area` names
(Newberry, `END_DATE` 2000, same method as T-002). Two rows name no county: the marker county is an inference (`coord_inferred = true`). The points mark the area; they are not its extent.

In [12]:
STATE_CESSIONS = {   # R1 id -> modern NY counties used for the marker
    'T1785NY': (['BROOME'], True,  'R1 names only the "present Binghamton area" (Broome Co.); the cession was larger.'),
    'T1788PG': (['ONTARIO', 'YATES', 'STEUBEN', 'MONROE'], False, 'Counties as listed in R1 present_day_area.'),
    'T1788ON': (['ONONDAGA', 'CORTLAND', 'OSWEGO'], False, 'Counties as listed in R1 present_day_area.'),
    'T1788OE': (['ONEIDA'], True,  'R1 names no county ("Central New York; ... near Oneida Lake"); Oneida Co. is a marker only.'),
    'T1789CA': (['CAYUGA', 'SENECA'], False, 'Counties as listed in R1 present_day_area.'),
}
cty = gpd.read_file(COUNTIES)
cty = cty[(cty.STATE_TERR == 'New York') & (cty.END_DATE.dt.year == 2000)].to_crs(5070)
sc = r1[r1.id.isin(STATE_CESSIONS)].copy()
assert len(sc) == len(STATE_CESSIONS) and sc.royce_cession_numbers.isna().all()
geoms, counties, inferred, notes = [], [], [], []
for rid in sc.id:
    names, inf, note = STATE_CESSIONS[rid]
    sel = cty[cty.NAME.isin(names)]; assert len(sel) == len(names), (rid, names)
    geoms.append(gpd.GeoSeries([sel.union_all().centroid], crs=5070).to_crs(4326).iloc[0])
    counties.append(';'.join(names)); inferred.append(inf); notes.append(note)
sc['geometry'] = geoms
sc['kind'] = 'state_cession_point'; sc['source'] = 'not_royce'; sc['dataset'] = DATASET['r1']
sc['year'] = sc.date_signed.str[:4].astype(int)
sc['date_precision'] = np.where(sc.date_signed.str.len() == 10, 'day', np.where(sc.date_signed.str.len() == 7, 'month', 'year'))
sc['year_flag'] = None; sc['todo'] = False
sc['coord_method'] = 'county_centroid'; sc['coord_counties'] = counties; sc['coord_inferred'] = inferred; sc['coord_note'] = notes
sc['coord_source'] = 'Newberry AHCB county polygons (modern extent, END_DATE 2000), data/raw/gis/newberry-ahcb-counties'
sc['lat'] = sc.geometry.apply(lambda p: round(p.y, 4)); sc['lon'] = sc.geometry.apply(lambda p: round(p.x, 4))
print('other R1 rows with no Royce number (not drawn here):', r1[r1.royce_cession_numbers.isna() & ~r1.id.isin(STATE_CESSIONS)].id.tolist())
sc[['id', 'name', 'date_signed', 'year', 'date_precision', 'native_parties', 'coord_counties', 'coord_inferred', 'lat', 'lon']]

other R1 rows with no Royce number (not drawn here): ['P1763', 'T1768HL', 'T1768FS', 'T1770', 'T1774', 'T1775', 'T1777D', 'T1777L', 'T1783GC', 'T1783GK', 'T1784PA', 'T1785PA', 'T1785FM', 'L1785', 'T1786S', 'H1786', 'N1787', 'T1789FH', 'R1830']


,id,name,date_signed,year,date_precision,native_parties,coord_counties,coord_inferred,lat,lon
15,T1785NY,Treaty of Fort Herkimer,1785-06,1785,month,Oneida; Tuscarora,BROOME,True,42.1602,-75.8194
20,T1788PG,Phelps–Gorham Purchase (First Treaty of Buffalo Creek),1788-07-08,1788,day,Seneca (participation of other Haudenosaunee nations unverified),ONTARIO;YATES;STEUBEN;MONROE,False,42.6249,-77.3986
21,T1788ON,New York–Onondaga treaty,1788-09-12,1788,day,Onondaga,ONONDAGA;CORTLAND;OSWEGO,False,43.1007,-76.1445
22,T1788OE,New York–Oneida treaty,1788-09-22,1788,day,Oneida,ONEIDA,True,43.2415,-75.4359
24,T1789CA,New York–Cayuga treaty,1789-02-25,1789,day,Cayuga,CAYUGA;SENECA,False,42.8691,-76.6477


## 3. Pre-1784 lines (T-022, R11) → segments
`research/pre1784_lines_R11.csv`: seven lines as numbered vertices, each `described` (the text names the point or the river) or `extrapolated` (computed from a bearing or joined between anchors), with `uncertainty_km`.
- **One feature per segment** (vertex to vertex). A segment takes the **weaker method** and the **larger uncertainty** of its two ends, floored at 3 km (most vertices are not GNIS-checked, R11). Bucket: low ≤ 10 km < medium ≤ 25 km < high.
- **Rivers:** where both ends are `described` and R11 says the line follows a named river, the segment follows the Natural Earth 10m river (read remotely) between the two vertices, if NE has that river and both vertices lie within 5 km of it. Otherwise it is straight; `path` says which and why (Lycoming, Towanda, Unadilla, Little River and the North Fork Kentucky are not in NE 10m).
- **augusta_1773 seq 4** has no anchor in any source read (R11). It is kept only because the 1773 Ceded Lands cannot be closed without it (`USE_A73_SEQ4`), and its segment is flagged `closure_assumed`.
- `proclamation_1763` is a reference line only (`role = reference`): it lies east of the zone edge except in Georgia (R11).


In [13]:
import shapely.ops
R11_CSV = config.PROJECT_DIR / 'research/pre1784_lines_R11.csv'
U_FLOOR_KM = 3            # T-022: floor on every segment (most R11 vertices are not GNIS-checked)
SNAP_KM = 5               # a described vertex must lie this close to the Natural Earth river, else the segment stays straight
USE_A73_SEQ4 = True       # augusta_1773 seq 4 has no source anchor (R11); used only because the Ceded Lands cannot close without it (flagged)
def bucket(u): return 'low' if u <= 10 else ('medium' if u <= 25 else 'high')   # T-022: low <= 10 km < medium <= 25 km < high
LINES = {   # line -> (name, year, R1 id); R1 has no row for the Augusta lines
    'proclamation_1763':  ('Royal Proclamation of 1763 (Atlantic divide, reference only)', 1763, 'P1763'),
    'augusta_1763_creek': ('Treaty of Augusta 1763 (Creek line)', 1763, None),
    'hard_labour_1768':   ('Treaty of Hard Labour 1768 (Cherokee line)', 1768, 'T1768HL'),
    'fort_stanwix_1768':  ('Treaty of Fort Stanwix 1768 (Six Nations line)', 1768, 'T1768FS'),
    'lochaber_1770':      ('Treaty of Lochaber 1770 (Cherokee line)', 1770, 'T1770'),
    'donelson_1771':      ('Donelson line 1771 (Lochaber line as run)', 1771, 'T1770'),
    'augusta_1773':       ('Treaty of Augusta 1773 (Ceded Lands)', 1773, None),
}
# Stretches where R11 names the river the line follows: (seq from, seq to, river as named, Natural Earth name or None = not in NE 10m)
RIVERS = {
    'fort_stanwix_1768':  [(1, 21, 'Ohio', 'Ohio'), (21, 23, 'Allegheny', 'Allegheny'), (24, 32, 'West Branch Susquehanna', 'W. Branch Susquehanna'),
                           (32, 35, 'Lycoming Creek', None), (36, 39, 'Towanda Creek', None), (39, 41, 'East Branch Susquehanna', 'Susquehanna'),
                           (42, 43, 'Delaware West Branch', 'Delaware'), (44, 47, 'Unadilla', None)],
    'donelson_1771':      [(2, 5, 'North Fork Kentucky', None), (5, 9, 'Kentucky', 'Kentucky'), (9, 15, 'Ohio', 'Ohio')],
    'augusta_1763_creek': [(1, 2, 'Little River', None), (3, 4, 'Briar Creek', None), (5, 6, 'Ogeechee', 'Ogeechee')],
    'augusta_1773':       [(1, 2, 'Ogeechee', 'Ogeechee')],
}
NE_RIVERS = [f'/vsizip//vsicurl/https://naciscdn.org/naturalearth/10m/physical/{k}.zip/{k}.shp' for k in ('ne_10m_rivers_lake_centerlines', 'ne_10m_rivers_north_america')]

r11 = pd.read_csv(R11_CSV)
assert not r11.duplicated(['line', 'seq']).any() and r11[['line', 'seq', 'lat', 'lon', 'method', 'uncertainty_km', 'anchor', 'source']].notna().all().all()
assert set(r11.method) <= {'described', 'mapped', 'extrapolated'} and set(r11.line) == set(LINES) and (r11.uncertainty_km > 0).all()
assert r11.lon.between(EXTENT[0], EXTENT[2]).all() and r11.lat.between(EXTENT[1], EXTENT[3]).all()
r11 = r11.sort_values(['line', 'seq']).reset_index(drop=True)
assert all((g.seq.diff().dropna() == 1).all() for _, g in r11.groupby('line'))
dropped = r11[(r11.line == 'augusta_1773') & (r11.seq == 4)] if not USE_A73_SEQ4 else r11.iloc[:0]
V = {(r.line, r.seq): r for r in r11.drop(dropped.index).itertuples()}
print('R11 vertices:', len(r11), '| by line', r11.line.value_counts().to_dict(), '| by method', r11.method.value_counts().to_dict(), '| dropped', len(dropped))

def short(anchor): return anchor.split(';')[0].strip()     # anchor text up to the first ';' (used in notes)
def cite(a, b): return a if b in a else b if a in b else f'{a}; {b}'
def km(g): return gpd.GeoSeries([g], crs=4326).to_crs(5070).length.iloc[0] / 1e3
def pt_km(p, g): return gpd.GeoSeries([p], crs=4326).to_crs(5070).distance(gpd.GeoSeries([g], crs=4326).to_crs(5070)).iloc[0] / 1e3
def seg_rule(va, vb):
    """T-022: a segment takes the weaker method and the larger uncertainty of its ends, floored at U_FLOOR_KM."""
    rank = {'described': 0, 'mapped': 1, 'extrapolated': 2}
    m = max([va.method, vb.method], key=rank.get)
    return m, float(max(va.uncertainty_km, vb.uncertainty_km, U_FLOOR_KM))

# Natural Earth 10m rivers, read remotely (as E01 reads NE land); nothing written
need = sorted({ne for runs in RIVERS.values() for *_, ne in runs if ne} | {'Savannah'})   # Savannah: only for the closure if USE_A73_SEQ4 is False
riv = pd.concat([gpd.read_file(u, bbox=tuple(EXTENT)) for u in NE_RIVERS], ignore_index=True)
riv = riv[riv.name.isin(need) & riv.featurecla.isin(['River', 'Lake Centerline'])]
RIV = {n: list(shapely.get_parts(shapely.line_merge(shapely.union_all(riv[riv.name == n].geometry.values)))) for n in need}
print('NE rivers used:', {n: len(p) for n, p in RIV.items()})

def river_path(a, b, ne):
    """Coords from a to b along the NE river part nearest both, or None (+ reason) if a vertex is more than SNAP_KM off or the path is implausible."""
    if ne not in RIV: return None, 'river not in Natural Earth 10m'
    best = min(RIV[ne], key=lambda p: max(pt_km(a, p), pt_km(b, p)))
    da, db = pt_km(a, best), pt_km(b, best)
    if max(da, db) > SNAP_KM: return None, f'vertex {max(da, db):.1f} km off the NE river'
    sub = shapely.ops.substring(best, best.project(a), best.project(b))
    path = shapely.LineString([a.coords[0]] + list(sub.coords)[1:-1] + [b.coords[0]])
    if km(path) > 4 * km(shapely.LineString([a, b])): return None, 'NE path > 4x the chord'
    return path, f'Natural Earth 10m river "{ne}" (vertices {da:.1f} / {db:.1f} km off)'

rows = []
for line, g in r11.drop(dropped.index).groupby('line', sort=False):
    vs = list(g.itertuples())
    for va, vb in zip(vs, vs[1:]):
        a, b = shapely.Point(va.lon, va.lat), shapely.Point(vb.lon, vb.lat)
        method, u = seg_rule(va, vb)
        run = next(((n, ne) for s0, s1, n, ne in RIVERS.get(line, []) if s0 <= va.seq and vb.seq <= s1), None)
        geom, path = shapely.LineString([a, b]), 'straight'
        if run and va.method == vb.method == 'described':
            p, why = river_path(a, b, run[1])
            if p is not None: geom, path = p, why
            else: path = f'straight ({run[0]}: {why})'
        elif run: path = f'straight ({run[0]}: an end is {method})'
        rows.append(dict(id=f'{line}:{va.seq:02d}-{vb.seq:02d}', line=line, line_name=LINES[line][0], seq_from=va.seq, seq_to=vb.seq,
                         anchor_from=va.anchor, anchor_to=vb.anchor, method=method, uncertainty_km=u, uncertainty=bucket(u), path=path,
                         river=run[0] if run else None, citation=cite(va.source, vb.source),
                         closure_assumed=bool(line == 'augusta_1773' and vb.seq == 4), geometry=geom))
pl = gpd.GeoDataFrame(rows, geometry='geometry', crs=4326)
pl['year'] = pl.line.map(lambda l: LINES[l][1]); pl['r1_id'] = pl.line.map(lambda l: LINES[l][2])
pl['kind'] = 'boundary_line'; pl['source'] = 'not_royce'; pl['year_flag'] = None; pl['todo'] = False; pl['flag'] = None
pl['dataset'] = 'R11 pre-1784 lines (research/pre1784_lines_R11.csv); river stretches: Natural Earth 10m rivers (public domain)'
pl['length_km'] = pl.to_crs(5070).length.div(1e3).round(1)
pl['todo_note'] = np.where(pl.closure_assumed, 'R11: seq 4 has no anchor in any source read; assumed join, used only to close the 1773 Ceded Lands.', None)
display(pl.groupby('line').agg(segments=('id', 'size'), on_river=('path', lambda s: int(s.str.startswith('Natural').sum())),
                                 extrapolated=('method', lambda s: int((s == 'extrapolated').sum())), km=('length_km', 'sum'), max_u=('uncertainty_km', 'max')))
display(pl[pl.river.notna()][['id', 'river', 'path']])


R11 vertices: 107 | by line {'fort_stanwix_1768': 48, 'proclamation_1763': 24, 'donelson_1771': 15, 'augusta_1763_creek': 7, 'hard_labour_1768': 6, 'augusta_1773': 4, 'lochaber_1770': 3} | by method {'described': 70, 'extrapolated': 37} | dropped 0


NE rivers used: {'Allegheny': 1, 'Delaware': 1, 'Kentucky': 1, 'Ogeechee': 1, 'Ohio': 4, 'Savannah': 3, 'Susquehanna': 1, 'W. Branch Susquehanna': 1}


,segments,on_river,extrapolated,km,max_u
line,,,,,
augusta_1763_creek,6,0,4,307.9,30.0
augusta_1773,3,0,2,134.6,40.0
donelson_1771,14,10,4,865.0,30.0
fort_stanwix_1768,47,32,4,2162.0,25.0
hard_labour_1768,5,0,3,607.7,12.0
lochaber_1770,2,0,1,371.6,10.0
proclamation_1763,23,0,23,1312.0,10.0


,id,river,path
0,augusta_1763_creek:01-02,Little River,straight (Little River: river not in Natural Earth 10m)
2,augusta_1763_creek:03-04,Briar Creek,straight (Briar Creek: an end is extrapolated)
4,augusta_1763_creek:05-06,Ogeechee,straight (Ogeechee: an end is extrapolated)
6,augusta_1773:01-02,Ogeechee,straight (Ogeechee: an end is extrapolated)
10,donelson_1771:02-03,North Fork Kentucky,straight (North Fork Kentucky: an end is extrapolated)
11,donelson_1771:03-04,North Fork Kentucky,straight (North Fork Kentucky: an end is extrapolated)
12,donelson_1771:04-05,North Fork Kentucky,straight (North Fork Kentucky: an end is extrapolated)
13,donelson_1771:05-06,Kentucky,"Natural Earth 10m river ""Kentucky"" (vertices 1.0 / 1.8 km off)"
14,donelson_1771:06-07,Kentucky,"Natural Earth 10m river ""Kentucky"" (vertices 1.8 / 0.7 km off)"
15,donelson_1771:07-08,Kentucky,"Natural Earth 10m river ""Kentucky"" (vertices 0.7 / 1.6 km off)"


## 3b. "Taken before 1784" zone (T-021, T-022, R10 §3, R11)
**Edge**, north to south (R11 'Which lines bound the zone'): Fort Stanwix 1768 from Wood Creek to the Kanawha mouth → Donelson 1771 down the Ohio and up the Kentucky to Long Island → Lochaber 1770 east along 36° 30′ → Hard Labour 1768 from the Lochaber corner to Tryon Mountain and the Savannah → join to augusta_1773 seq 4 → Augusta 1773 to the Lower Creek Path on the Ogeechee → Augusta 1763 to Santa Savilla.
**Closures (assumed, no source line):** due north from Wood Creek and due south from Santa Savilla (the 1763 article ends "Southward as far as Georgia extends"); the zone is then cut to the US of 1784 (E01 `borders.topojson`, 1784 snapshot: removes Canada and Spanish Florida), to E01's land mask (NE land read remotely minus the `water.topojson` lakes) and minus every Royce polygon (all dates).
**Pieces:** each run of edge segments with the same line and bucket gets a band of ± its `uncertainty_km` inside the zone; overlaps go to the higher bucket. The two closure regions (New York north of Wood Creek; Georgia south of Santa Savilla) are `high`. The rest is one `interior` piece (`low`). Each piece has a `note`; every extrapolated stretch is named ("Line extrapolated between <a> and <b>; position uncertain by ~N km").


In [14]:
NE_LAND_URL = 'https://naciscdn.org/naturalearth/10m/physical/ne_10m_land.zip'
NE_LAND_LOCAL = GIS / 'natural-earth-10m-land' / 'ne_10m_land' / 'ne_10m_land.shp'
ne_land = gpd.read_file(NE_LAND_LOCAL if NE_LAND_LOCAL.exists() else f'/vsizip//vsicurl/{NE_LAND_URL}/ne_10m_land.shp', bbox=tuple(EXTENT))
LAND = shapely.union_all(shapely.make_valid(ne_land.geometry.values)).intersection(EXT_BOX)
lakes = [g for g, _ in topo_decode(json.loads((WEB_DIR / 'water.topojson').read_text()), 'lakes') if g is not None]
LAND_MASK = shapely.make_valid(LAND.difference(shapely.union_all(shapely.make_valid(np.asarray(lakes)))))   # = E01 LAND_MASK
ROYCE_ALL = shapely.union_all(shapely.make_valid(np.asarray(list(roy.geometry[roy.has_geometry]))))
OUTSIDE_ROYCE = shapely.make_valid(LAND_MASK.difference(ROYCE_ALL))
def km2(g): return gpd.GeoSeries([g], crs=4326).to_crs(5070).area.iloc[0] / 1e6

PAGE_NOTE = "Cessions of 1775–83 (Henderson, DeWitt's Corner, Long Island of Holston, Augusta 1783) are not yet drawn."
CLOSURE_KM = 40   # band width for the assumed join from Towatuhe (R11's figure for augusta_1773 seq 4); a band width, not an estimate
S = pl.set_index('id')

# Zone edge, north to south (R11 'Which lines bound the zone'): (line, seq from, seq to); a segment run backwards is reversed
# 'join' = an assumed closure between two R11 vertices that no source connects
EDGE = [('fort_stanwix_1768', 48, 15), ('donelson_1771', 15, 1), ('lochaber_1770', 2, 1), ('hard_labour_1768', 'L1', 1)] + (
       [('join', ('hard_labour_1768', 1), ('augusta_1773', 4)), ('augusta_1773', 4, 1), ('augusta_1763_creek', 5, 7)] if USE_A73_SEQ4 else
       [('join', ('hard_labour_1768', 1), ('augusta_1763_creek', 1)), ('augusta_1763_creek', 1, 7)])   # without seq 4: down the Savannah, 1773 Ceded Lands left out

def edge_pieces():
    """-> list of dicts (one per edge piece, north to south) with geometry and the attributes of the segment it comes from."""
    out = []
    def add(sid, geom, **kw):
        r = S.loc[sid]
        out.append(dict(seg_id=sid, line=r.line, line_name=r.line_name, a=short(r.anchor_from), b=short(r.anchor_to), method=r.method,
                        u=r.uncertainty_km, closure_assumed=bool(r.closure_assumed), geometry=geom) | kw)
    for line, s0, s1 in EDGE:
        if line == 'join':     # Towatuhe (Hard Labour's south end) to the next Georgia line: straight, or down the NE Savannah when joining the 1763 line
            va, vb = V[s0], V[s1]; a, b = shapely.Point(va.lon, va.lat), shapely.Point(vb.lon, vb.lat)
            geom = river_path(a, b, 'Savannah')[0] if not USE_A73_SEQ4 else None
            out.append(dict(seg_id=f'join:{s0[0]}:{s0[1]:02d}-{s1[0]}:{s1[1]:02d}', line='closure', line_name='Assumed join', a=short(va.anchor), b=short(vb.anchor),
                            method='extrapolated', u=float(CLOSURE_KM), closure_assumed=True, geometry=geom if geom is not None else shapely.LineString([a, b])))
            continue
        if s0 == 'L1':         # Lochaber starts on the Hard Labour Tryon -> Chiswell line: take that line from Lochaber seq 1 back to Tryon Mountain
            l1, hl4 = V[('lochaber_1770', 1)], V[('hard_labour_1768', 4)]
            seg = S.loc['hard_labour_1768:04-05'].geometry; p = shapely.Point(l1.lon, l1.lat)
            assert pt_km(p, seg) < 1, pt_km(p, seg)
            m, u = seg_rule(l1, hl4)
            out.append(dict(seg_id='hard_labour_1768:04-05', line=line, line_name=LINES[line][0], a=short(l1.anchor), b=short(hl4.anchor), method=m, u=u,
                            closure_assumed=False, partial='from lochaber_1770 seq 1 to Tryon Mountain', geometry=shapely.LineString([p, (hl4.lon, hl4.lat)])))
            s0 = 4
        step = -1 if s1 < s0 else 1
        for s in range(s0, s1, step):
            lo, hi = sorted((s, s + step))
            sid = f'{line}:{lo:02d}-{hi:02d}'; g_ = S.loc[sid].geometry
            add(sid, g_.reverse() if step < 0 else g_)
            if step < 0: out[-1]['a'], out[-1]['b'] = out[-1]['b'], out[-1]['a']
    return out

EP = edge_pieces()
coords = [c for e in EP for c in e['geometry'].coords]
for e, f in zip(EP, EP[1:]): assert e['geometry'].coords[-1] == f['geometry'].coords[0], (e['seg_id'], f['seg_id'])
edge_line = shapely.LineString(coords)
assert edge_line.is_simple, 'zone edge crosses itself'
n0, s0_ = coords[0], coords[-1]
# closures: due north from Wood Creek and due south from Santa Savilla, then round the east of the extent; cut to the US of 1784 below
ring = [(n0[0], EXTENT[3] + 5)] + coords + [(s0_[0], EXTENT[1] - 5), (EXTENT[2] + 5, EXTENT[1] - 5), (EXTENT[2] + 5, EXTENT[3] + 5)]
ZONE_POLY = shapely.Polygon(ring); assert ZONE_POLY.is_valid

# US of 1784 (31 Dec snapshot in E01's borders.topojson: the states + Vermont + federal territory): removes Canada and Spanish Florida.
# Closed by +/- 0.02 deg so the seams between E01's per-feature simplified states do not cut slivers.
B_states = topo_decode(json.loads((WEB_DIR / 'borders.topojson').read_text()), 'states')
st84 = [(g_, p) for g_, p in B_states if 1784 in p['years']]
US1784 = shapely.union_all(shapely.make_valid(np.asarray([g_ for g_, _ in st84]))).buffer(0.02).buffer(-0.02)
NY1784 = shapely.make_valid(next(g_ for g_, p in st84 if p['name'] == 'New York'))
print('1784 states/territories:', sorted(p['name'] for _, p in st84))

def polys(g_):
    ps = [p for p in shapely.get_parts(shapely.make_valid(g_)) if p.geom_type == 'Polygon' and p.area > 1e-10] if g_ is not None and not g_.is_empty else []
    return shapely.MultiPolygon(ps) if ps else None

zone = polys(OUTSIDE_ROYCE.intersection(ZONE_POLY).intersection(US1784))

def band(geom, u_km):
    return gpd.GeoSeries([geom.segmentize(0.05)], crs=4326).to_crs(5070).buffer(u_km * 1e3).to_crs(4326).iloc[0]

# Stretches: consecutive edge pieces with the same line and bucket
for e in EP: e['bucket'] = bucket(e['u'])
stretches = []
for e in EP:
    if stretches and stretches[-1][0]['line'] == e['line'] and stretches[-1][0]['bucket'] == e['bucket'] and not e['closure_assumed'] and not stretches[-1][0]['closure_assumed']:
        stretches[-1].append(e)
    else: stretches.append([e])

def stretch_note(st):
    ex = [e for e in st if e['method'] == 'extrapolated']
    umax = max(e['u'] for e in st)
    head = f"{st[0]['line_name']}, {st[0]['a']} to {st[-1]['b']}: position uncertain by up to ~{umax:g} km."
    if any(e['closure_assumed'] for e in st):
        head = (f"Closure assumed between {st[0]['a']} and {st[-1]['b']}: no source names this stretch (R11), "
                f"so the zone is closed with a join; position uncertain by ~{umax:g} km or more.")
    return ' '.join([head] + [f"Line extrapolated between {e['a']} and {e['b']}; position uncertain by ~{e['u']:g} km." for e in ex if not e['closure_assumed']])

RANK = {'high': 0, 'medium': 1, 'low': 2}
piece_defs = []
lat_n, lat_s = n0[1], s0_[1]
piece_defs.append(dict(kind_='closure_north', uncertainty='high', uncertainty_km=None, closure_assumed=True, edge_line='closure', edge_from=short(V[('fort_stanwix_1768', 48)].anchor), edge_to='1783 boundary',
    region=NY1784.intersection(shapely.box(EXTENT[0], lat_n, EXTENT[2], EXTENT[3] + 5)),
    note=('Closure assumed: the 1768 deed stops at Canada Creek on Wood Creek and names no line north of it. Northern New York is included by closing the zone '
          'due north to the 1783 boundary; no source draws that line. Parts of this land were sold to New York State after 1784 (e.g. Oneida 1788, R1), so this piece may be too large.')))
piece_defs.append(dict(kind_='closure_south', uncertainty='high', uncertainty_km=None, closure_assumed=True, edge_line='closure', edge_from=short(V[('augusta_1763_creek', 7)].anchor), edge_to='1783 Georgia-Florida boundary',
    region=shapely.box(EXTENT[0] - 5, EXTENT[1] - 5, EXTENT[2] + 5, lat_s),
    note=("Closure assumed: the 1763 Augusta article ends at Santa Savilla with 'Southward as far as Georgia extends'. "
          'Drawn due south to the 1783 Georgia-Florida boundary; the course is not described.')))
for i, st in sorted(enumerate(stretches), key=lambda t: (RANK[t[1][0]['bucket']], t[0])):
    piece_defs.append(dict(kind_='edge_band', uncertainty=st[0]['bucket'], uncertainty_km=max(e['u'] for e in st), closure_assumed=any(e['closure_assumed'] for e in st),
        edge_line=st[0]['line'], edge_from=st[0]['a'], edge_to=st[-1]['b'], edge_segments=';'.join(e['seg_id'] for e in st), stretch=i,
        n_extrapolated=sum(e['method'] == 'extrapolated' for e in st),
        region=shapely.union_all([band(e['geometry'], e['u']) for e in st]), note=stretch_note(st)))
piece_defs.sort(key=lambda d: RANK[d['uncertainty']])   # stable: closures first within 'high'

taken, pieces = None, []
for d in piece_defs:
    g_ = zone.intersection(d['region'])
    if taken is not None: g_ = g_.difference(taken)
    g_ = polys(g_)
    if g_ is None: continue
    taken = g_ if taken is None else shapely.union_all([taken, g_])
    pieces.append({k: v for k, v in d.items() if k != 'region'} | dict(geometry=g_))
interior = polys(zone.difference(taken))
pieces.append(dict(kind_='interior', uncertainty='low', uncertainty_km=None, closure_assumed=False, edge_line=None, edge_from=None, edge_to=None,
                   note='Inside the pre-1784 treaty lines. The uncertainty marks only where the bounding line runs; this land lies well inside it.', geometry=interior))
pc = pd.DataFrame(pieces).rename(columns={'kind_': 'piece'}).drop(columns='stretch')
pc['area_km2'] = [round(km2(g_)) for g_ in pc.geometry]

# line roles (T-022): which segments bound the zone
edge_ids = {e['seg_id'] for e in EP if e['line'] != 'closure'}
pl['zone_edge'] = pl.id.isin(edge_ids)
pl['role'] = np.select([pl.line == 'proclamation_1763', pl.zone_edge], ['reference', 'zone_edge'], 'not_edge')
pl.loc[pl.id == 'hard_labour_1768:04-05', 'role'] = 'zone_edge_part'     # only Lochaber seq 1 -> Tryon Mountain bounds the zone

LABEL = 'Taken before 1784: not mapped by Royce'
b84 = pc.assign(id=[f'before_1784:{i:02d}' for i in range(len(pc))], label=LABEL, kind='before_1784_zone', source='not_royce',
    dataset=('Natural Earth 10m land (public domain) minus water.topojson lakes, minus all Royce polygons (USFS, CC BY 4.0), inside the US of 1784 '
             '(E01 borders.topojson, Newberry AHCB), east of the R11 pre-1784 lines (research/pre1784_lines_R11.csv)'),
    west_bound='fort_stanwix_1768 > donelson_1771 > lochaber_1770 > hard_labour_1768 > augusta_1773 > augusta_1763_creek (R11); proclamation_1763 is a reference line only',
    page_note=PAGE_NOTE, year=None, year_flag='before_1784', todo=False, todo_note=None, flag=np.where(pc.closure_assumed, 'closure_assumed', None))
b84 = b84[['id'] + [c for c in b84.columns if c not in ('id', 'geometry')] + ['geometry']]
zone_km2 = km2(zone)
print(f'before_1784: {zone_km2:,.0f} km2 in {len(zone.geoms)} parts -> {len(b84)} pieces | edge pieces {len(EP)} in {len(stretches)} stretches | lines on the edge {int(pl.zone_edge.sum())}')
display(b84.groupby('uncertainty').agg(pieces=('id', 'size'), km2=('area_km2', 'sum'), closures=('closure_assumed', 'sum')).assign(share=lambda d: (d.km2 / d.km2.sum()).round(3)))
display(b84.drop(columns=['geometry', 'dataset', 'west_bound', 'page_note', 'label'])[['id', 'piece', 'uncertainty', 'uncertainty_km', 'closure_assumed', 'edge_line', 'area_km2', 'note']])

# diagnostic only: land outside every Royce polygon, by part (T-021)
op = gpd.GeoDataFrame(geometry=[p for p in shapely.get_parts(OUTSIDE_ROYCE) if p.geom_type == 'Polygon'], crs=4326)
op['km2'] = op.to_crs(5070).area / 1e6
print(f'land mask {km2(LAND_MASK):,.0f} km2 | outside every Royce polygon {op.km2.sum():,.0f} km2 in {len(op)} parts ({int((op.km2 > 1000).sum())} > 1,000 km2) | in the zone {zone_km2 / op.km2.sum():.0%}')


1784 states/territories: ['Connecticut', 'Delaware', 'Erie Tri. Unorg. FT', 'Georgia', 'Maryland', 'Massachusetts', 'New Hampshire', 'New Jersey', 'New York', 'North Carolina', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'Unorg. Fed. Terr.', 'Vermont Republic', 'Virginia']


before_1784: 764,053 km2 in 69 parts -> 18 pieces | edge pieces 58 in 16 stretches | lines on the edge 57


,pieces,km2,closures,share
uncertainty,,,,
high,6,49593,4,0.065
low,7,709312,0,0.928
medium,5,5153,0,0.007


,id,piece,uncertainty,uncertainty_km,closure_assumed,edge_line,area_km2,note
0,before_1784:00,closure_north,high,NaN,True,closure,32930,Closure assumed: the 1768 deed stops at Canada Creek on Wood Creek and names no line n...
1,before_1784:01,closure_south,high,NaN,True,closure,2689,Closure assumed: the 1763 Augusta article ends at Santa Savilla with 'Southward as far...
2,before_1784:02,edge_band,high,30.0,False,donelson_1771,4119,"Donelson line 1771 (Lochaber line as run), North Fork Kentucky River at Hazard KY (ass..."
3,before_1784:03,edge_band,high,40.0,True,closure,2254,"Closure assumed between Towatuhe, north bank of Savannah River (placed on the N50E bea..."
4,before_1784:04,edge_band,high,40.0,True,augusta_1773,1557,Closure assumed between Savannah River near Towatuhe (assumed join with the 1766 SC-Ch...
5,before_1784:05,edge_band,high,30.0,False,augusta_1763_creek,6044,"Treaty of Augusta 1763 (Creek line), Lower Creek Path at the main stream of the Ogeech..."
6,before_1784:06,edge_band,medium,15.0,False,fort_stanwix_1768,1207,"Treaty of Fort Stanwix 1768 (Six Nations line), Mouth of Tianaderha (Unadilla River) a..."
7,before_1784:07,edge_band,medium,25.0,False,fort_stanwix_1768,2690,"Treaty of Fort Stanwix 1768 (Six Nations line), Head of Lycoming Creek (McNett Twp) to..."
8,before_1784:08,edge_band,medium,20.0,False,donelson_1771,1042,"Donelson line 1771 (Lochaber line as run), Three Forks of the Kentucky (Beattyville KY..."
9,before_1784:09,edge_band,medium,12.0,False,hard_labour_1768,35,"Treaty of Hard Labour 1768 (Cherokee line), Dewisses Corner or the Yellow Water (Due W..."


land mask 4,336,814 km2 | outside every Royce polygon 1,735,516 km2 in 464 parts (7 > 1,000 km2) | in the zone 44%


In [15]:
def feats(df, idcol):
    cols = [c for c in df.columns if c != 'geometry']
    return [(None if (g is None or (isinstance(g, float) and np.isnan(g))) else g, dict(zip(cols, row)), fid)
            for g, row, fid in zip(df.geometry, df[cols].itertuples(index=False, name=None), df[idcol])]

native = topo_encode({
    'royce_cessions': feats(roy, 'cessnum'),
    'state_cessions': feats(sc, 'id'),
    'pre1784_lines':  feats(pl, 'id'),      # T-022: R11 segments
    'before_1784':    feats(b84, 'id'),     # T-021, T-022: pieces by uncertainty
})
write_json(native, WEB_DIR / 'native_land.topojson')
print({k: len(v['geometries']) for k, v in native['objects'].items()}, '| arcs', len(native['arcs']))

wrote web/data/native_land.topojson  2.16 MB
{'royce_cessions': 450, 'state_cessions': 5, 'pre1784_lines': 100, 'before_1784': 18} | arcs 735


## 4. Reserves (R4 + T-002) → `reservations.json`
Every CSV column is kept. Added: `start_year`, `end_year` and their flags, parsed from the text dates. Rule: the first four-digit year;
flag `exact` for a plain date (`1797-09-15`, `1646`), `approx` for anything else (`c. 1705`, `1820s`, `1801–1802`, `1807 (or 1805)`).
No year when the text says the reserve continues, ended after 1840, or was only reduced; the flag says which. 7 rows have no point (`coord_method = none`, dispersed allotment classes or no place).

In [16]:
res = pd.read_csv(RES_CSV)
YEAR = re.compile(r'(1[5-9]\d\d)')
EXACT = re.compile(r'^\d{4}(-\d{2}(-\d{2})?)?$')
def parse(txt, end=False):
    if txt is None or (isinstance(txt, float) and np.isnan(txt)): return None, 'no_date'
    t = str(txt).strip(); low = t.lower()
    if end and 'continuing' in low: return None, 'continuing'
    if end and low.startswith('(after'): return None, 'after_1840'
    if end and 'reduced' in low: return None, 'reduced_only'
    m = YEAR.search(t)
    if not m: return None, 'unparsed'
    return int(m.group(1)), ('exact' if EXACT.match(t) else 'approx')
res[['start_year', 'start_flag']] = [parse(v) for v in res.date_established]
res[['end_year', 'end_flag']] = [parse(v, end=True) for v in res.date_ended]
res['start_year'] = res.start_year.astype('Int64'); res['end_year'] = res.end_year.astype('Int64')
res['has_coords'] = res.lat.notna() & res.lon.notna()
res['dataset'] = DATASET['r4']
res['year'] = res.start_year; res['year_flag'] = np.where(res.start_year.isna(), res.start_flag, None)
print('start flags', res.start_flag.value_counts().to_dict(), '| end flags', res.end_flag.value_counts().to_dict(), '| no coords', (~res.has_coords).sum())
res[(res.start_flag != 'exact') | (res.end_flag != 'exact')][['id', 'date_established', 'start_year', 'start_flag', 'date_ended', 'end_year', 'end_flag']]

start flags {'exact': 61, 'no_date': 3, 'approx': 2} | end flags {'exact': 26, 'no_date': 19, 'continuing': 11, 'approx': 5, 'after_1840': 3, 'reduced_only': 2} | no coords 7


,id,date_established,start_year,start_flag,date_ended,end_year,end_flag
0,R4-001,1646,1646,exact,NaN,<NA>,no_date
1,R4-002,NaN,<NA>,no_date,NaN,<NA>,no_date
3,R4-004,c. 1705,1705,approx,1824 (allotment begins); last parcels 1870s,1824,approx
5,R4-006,1758,1758,exact,1801–1802,1801,approx
6,R4-007,1788-09-22,1788,exact,1795–1846 (reduced),<NA>,reduced_only
7,R4-008,1788-09-12,1788,exact,(reduced 1793 and 1795),<NA>,reduced_only
8,R4-009,1789-02-25,1789,exact,1807 (or 1805),1807,approx
9,R4-010,mid-1780s,1780,approx,1820s,1820,approx
10,R4-011,1774-01-20,1774,exact,1820s–1830s,1820,approx
16,R4-017,1797-09-15,1797,exact,NaN,<NA>,no_date


In [17]:
write_json([clean_props(r) for r in res.to_dict('records')], WEB_DIR / 'reservations.json')
con.register('res_v', res)
q("""with s as (select (start_year::int // 10) * 10 as decade, count(*) started from res_v where start_year is not null group by 1),
          e as (select (end_year::int // 10) * 10 as decade, count(*) ended from res_v where end_year is not null group by 1)
     select decade, coalesce(started, 0) started, coalesce(ended, 0) ended from s full join e using (decade) order by decade""")

wrote web/data/reservations.json  0.09 MB


,decade,started,ended
0,1640,2,0
1,1680,1,0
2,1700,1,0
3,1730,1,0
4,1750,1,0
5,1760,1,0
6,1770,1,0
7,1780,4,0
8,1790,15,1
9,1800,1,2


## 5. Trail of Tears (NPS) → `removal_routes.topojson`
Three GeoJSON pages of one ArcGIS query (offsets 0, 1000, 2000), concatenated. All NPS columns kept. Undated: `year = null`, `year_flag = undated`.
`SOURCEDATE`, `CREATEDATE`, `EDITDATE` are NPS record dates in epoch ms (not event dates).

In [18]:
tt = pd.concat([gpd.read_file(p) for p in TRAIL], ignore_index=True)
tt = gpd.GeoDataFrame(tt, geometry='geometry', crs=4326)
print('pages', [p.name for p in TRAIL], '| features', len(tt), '| duplicate OBJECTID', tt.OBJECTID.duplicated().sum(), '| duplicate GlobalID', tt.GlobalID.duplicated().sum())
print('geometry', tt.geom_type.value_counts().to_dict(), '| empty', tt.is_empty.sum(), '| null', tt.geometry.isna().sum())
tt['kind'] = 'removal_route'; tt['source'] = 'nps_trte'; tt['dataset'] = DATASET['nps']
tt['year'] = None; tt['year_flag'] = 'undated'
display(tt.TRALTNAME.value_counts(dropna=False).to_frame('segments'))
display(tt.DATAACCESS.value_counts(dropna=False).to_frame('segments'))
# T-011: clip to the shared map extent
tt_n0 = len(tt)
cut = [clip_to_extent(g, 'LineString') for g in tt.geometry]
tt['geometry'] = [g for g, _ in cut]; tt['clipped_to_extent'] = [s == 'clipped' for _, s in cut]
print('trail_of_tears clip:', pd.Series([s for _, s in cut]).value_counts().to_dict())
tt = tt[[s != 'outside' for _, s in cut]].reset_index(drop=True)
print(f'trail_of_tears: {tt_n0} segments before -> {len(tt)} after')
routes = topo_encode({'trail_of_tears': feats(tt, 'OBJECTID')})
write_json(routes, WEB_DIR / 'removal_routes.topojson')

pages ['TRTE_NHT_off0.geojson', 'TRTE_NHT_off1000.geojson', 'TRTE_NHT_off2000.geojson'] | features 2006 | duplicate OBJECTID 0 | duplicate GlobalID 0
geometry {'LineString': 1381, 'MultiLineString': 625} | empty 0 | null 0


,segments
TRALTNAME,
Water Route,536
Northern Route,408
Roundup Route,268
Benge Route,228
Bell Route,199
Overland Water Route,100
Taylor Route,76
Hildebrand Route,70
Bell Route|Whiteley-Drane Route|Overland Water Route,47


,segments
DATAACCESS,
Unrestricted,1975
Internal NPS Only,31


trail_of_tears clip: {'inside': 2006}
trail_of_tears: 2006 segments before -> 2006 after


wrote web/data/removal_routes.topojson  4.08 MB


## 6. Assertions
Counts, unique keys, no null keys; every feature has a source and a year (or null + flag); the files read back and match the inputs within the quantization step.

In [19]:
nl = json.loads((WEB_DIR / 'native_land.topojson').read_text())
rr = json.loads((WEB_DIR / 'removal_routes.topojson').read_text())
rj = json.loads((WEB_DIR / 'reservations.json').read_text())
n_tbl = int((tbl[DATE_COLS].min(axis=1).dt.year <= WINDOW_END).sum())
G = nl['objects']['royce_cessions']['geometries']

# Royce: one feature per dated table row; polygons = the ones with a shape
assert len(G) == len(roy) == n_tbl - len(roy_out) == roy_n0 - len(roy_out), (len(G), n_tbl, len(roy_out))   # T-011: minus the rows wholly outside the extent
assert sum(g['type'] is not None for g in G) == roy.has_geometry.sum() == len(set(roy.cessnum) & set(shp.CESSNUM.dropna()))
assert all(g['type'] is None for g in G if g['properties']['flag'] == 'no_geometry')
ids = [g['id'] for g in G]; assert None not in ids and len(set(ids)) == len(ids)
assert all(g['properties']['cession_year'] <= WINDOW_END for g in G if g['properties']['cession_year'] is not None)
assert all(g['properties']['year_flag'] == 'no_later_usfs_cession' for g in G if g['properties']['cession_year'] is None)

# T-021: destinations dated as assignments
P = {g['id']: g['properties'] for g in G}
D_ = {k: p for k, p in P.items() if p['is_destination']}
assert len(D_) == len(dest) and set(D_) == set(dest.cessnum)
bad = [k for k, p in D_.items() if p['assigned_year'] is not None and p['cession_year'] is not None and p['cession_year'] <= p['assigned_year']]
assert not bad, bad                                                     # no destination is ceded in or before the year it was assigned
assert all(p['cession_year'] == p['cession_year_usfs'] and p['cession_date'] == p['cession_date_usfs'] for p in P.values() if p['redated'] in (None, 'todo_no_assigned_date'))
assert all(p['todo'] and p['todo_note'] for p in D_.values() if p['assigned_year'] is None)
assert all(p['assigned_verified'] in ('y', 'n') and p['nation_assigned'] for p in D_.values() if p['assigned_year'] is not None)
assert all(p['assigned_year'] is None and p['nation_assigned'] is None for p in P.values() if not p['is_destination'])
assert [k for k, p in P.items() if p['attr_flag']] == ['492'] and 'Santee Sioux' in P['492']['PRESDAYTRB']   # flagged, not changed
assert sum(p['redated'] == 'next_usfs_cession' for p in P.values()) == int((roy.redated == 'next_usfs_cession').sum())

# T-021/T-022: before_1784 pieces
B = nl['objects']['before_1784']['geometries']
BP = [g['properties'] for g in B]
assert len(B) == len(b84) > 1 and len({g['id'] for g in B}) == len(B) and None not in [g['id'] for g in B]
assert all(g['type'] in ('Polygon', 'MultiPolygon') for g in B)                      # no null pieces
assert all(p['label'] == 'Taken before 1784: not mapped by Royce' and p['note'] and p['page_note'] == PAGE_NOTE for p in BP)
assert {p['uncertainty'] for p in BP} <= {'low', 'medium', 'high'} and sum(p['piece'] == 'interior' for p in BP) == 1
assert all(bucket(p['uncertainty_km']) == p['uncertainty'] for p in BP if p['piece'] == 'edge_band')
assert all(p['uncertainty'] == 'high' and p['closure_assumed'] for p in BP if p['piece'].startswith('closure'))
dec_b = [g_ for g_, _ in topo_decode(nl, 'before_1784')]
zb = shapely.union_all(shapely.make_valid(np.asarray(dec_b, dtype=object)))     # quantized rings can touch: make valid before the union
assert zb.buffer(-0.01).intersection(ROYCE_ALL).area < 1e-6 and zb.difference(LAND_MASK.buffer(0.01)).area < 1e-6
assert zb.difference(US1784.buffer(0.01)).area < 1e-6 and zb.difference(ZONE_POLY.buffer(0.01)).area < 1e-6
sum_km2, union_km2 = sum(km2(g_) for g_ in b84.geometry), km2(shapely.union_all(list(b84.geometry)))
assert abs(sum_km2 - zone_km2) / zone_km2 < 2e-5 and abs(union_km2 - zone_km2) / zone_km2 < 2e-5, (sum_km2, union_km2, zone_km2)   # pieces tile the zone (overlaps/slivers < 0.002 %)
notes = ' '.join(p['note'] for p in BP)
unnamed = [(e['a'], e['b']) for e in EP if e['method'] == 'extrapolated' and not e['closure_assumed'] and f"between {e['a']} and {e['b']}" not in notes]
assert not unnamed, unnamed                                                       # every extrapolated edge stretch is named in a note

# T-022: pre1784_lines, one feature per R11 segment
L_ = nl['objects']['pre1784_lines']['geometries']; LP = {g['id']: g['properties'] for g in L_}
assert len(L_) == len(pl) == len(r11) - len(dropped) - r11.line.nunique() and len(LP) == len(L_) and set(LP) == set(pl.id)
assert all(g['type'] == 'LineString' for g in L_)
assert all(p['method'] in ('described', 'mapped', 'extrapolated') and p['uncertainty_km'] >= U_FLOOR_KM and bucket(p['uncertainty_km']) == p['uncertainty'] for p in LP.values())
assert all(p['year'] == LINES[p['line']][1] and p['citation'] for p in LP.values())
for (g_, p) in topo_decode(nl, 'pre1784_lines'):                                   # ends sit on the R11 vertices (within 2 quantization steps)
    a, b = V[(p['line'], p['seq_from'])], V[(p['line'], p['seq_to'])]
    step = max(nl['transform']['scale']) * 2.01
    assert shapely.Point(g_.coords[0]).distance(shapely.Point(a.lon, a.lat)) <= step and shapely.Point(g_.coords[-1]).distance(shapely.Point(b.lon, b.lat)) <= step, p['id']
    assert (p['method'] == 'extrapolated') == ('extrapolated' in (a.method, b.method)) and p['uncertainty_km'] == max(a.uncertainty_km, b.uncertainty_km, U_FLOOR_KM)
assert {k for k, p in LP.items() if p['zone_edge']} == edge_ids and not any(p['zone_edge'] for p in LP.values() if p['line'] == 'proclamation_1763')
assert all(p['role'] == 'reference' for p in LP.values() if p['line'] == 'proclamation_1763')
assert [k for k, p in LP.items() if p['closure_assumed']] == (['augusta_1773:03-04'] if USE_A73_SEQ4 else [])

# every feature in every file: a source and a year, or null year + flag
for name, obj in nl['objects'].items():
    for g in obj['geometries']:
        p = g['properties']; assert p['source'] and p['dataset'], (name, g.get('id'))
        assert p['year'] is not None or p.get('year_flag') or p.get('flag'), (name, g.get('id'))
assert {g['properties']['source'] for g in nl['objects']['state_cessions']['geometries']} == {'not_royce'}
for r in rj:
    assert r['id'] and r['source'] and r['dataset']
    assert r['start_year'] is not None or r['start_flag'] in ('no_date', 'unparsed')
    assert r['end_year'] is not None or r['end_flag'] in ('no_date', 'continuing', 'after_1840', 'reduced_only', 'unparsed')
assert len(rj) == len(pd.read_csv(RES_CSV)) and len({r['id'] for r in rj}) == len(rj)
assert sum(r['has_coords'] for r in rj) == res.has_coords.sum()
T = rr['objects']['trail_of_tears']['geometries']
assert len(T) == len(tt) and len({g['id'] for g in T}) == len(T) and None not in [g['id'] for g in T]
assert all(g['properties']['year'] is None and g['properties']['year_flag'] == 'undated' for g in T)

# every source column survives
assert set(tbl.columns.drop('_earliest')) | {'CESSNUMSOR', 'SHAPE_AREA', 'SHAPE_LEN'} <= set(G[0]['properties'])
assert set(r1.columns) <= set(nl['objects']['state_cessions']['geometries'][0]['properties'])
assert set(pd.read_csv(RES_CSV).columns) <= set(rj[0])
assert set(tt.columns.drop('geometry')) <= set(T[0]['properties'])
assert len(T) == tt_n0 - int(sum(s == 'outside' for _, s in cut))

# geometry round trip: decoded shapes within ~2 quantization steps of the inputs
def max_dev(topo, name, src):
    step = max(topo['transform']['scale']) * 2.01
    dev = [a.hausdorff_distance(b) for (a, _), b in zip(topo_decode(topo, name), src) if b is not None]
    return max(dev), step
d, s = max_dev(nl, 'royce_cessions', list(roy.geometry.where(roy.has_geometry, None))); assert d <= s, (d, s); print(f'royce max deviation {d:.5f} deg (limit {s:.5f})')
d, s = max_dev(nl, 'state_cessions', list(sc.geometry)); assert d <= s
d, s = max_dev(rr, 'trail_of_tears', list(tt.geometry)); assert d <= s, (d, s); print(f'trail max deviation {d:.5f} deg (limit {s:.5f})')
d, s = max_dev(nl, 'pre1784_lines', list(pl.geometry)); assert d <= s, (d, s)                       # T-022
d, s = max_dev(nl, 'before_1784', list(b84.geometry)); assert d <= s, (d, s); print(f'before_1784 max deviation {d:.5f} deg (limit {s:.5f})')

# Reference decoder (topojson-client, node): same feature counts, nulls only where intended,
# every ring >= 4 closed positions, every line >= 2 positions
chk = topojson_client_check([WEB_DIR / 'native_land.topojson', WEB_DIR / 'removal_routes.topojson'])
display(chk)
want_n = {('native_land.topojson', 'royce_cessions'): len(roy), ('native_land.topojson', 'state_cessions'): len(sc),
          ('native_land.topojson', 'pre1784_lines'): len(pl), ('native_land.topojson', 'before_1784'): len(b84), ('removal_routes.topojson', 'trail_of_tears'): len(tt)}
assert {(r.file, r.object): r.features for r in chk.itertuples()} == want_n, chk
c = chk.set_index('object')
assert c.loc['royce_cessions', 'null'] == (~roy.has_geometry).sum() and c.loc['pre1784_lines', 'null'] == 0 and c.loc['pre1784_lines', 'lines'] == len(pl)
assert c.loc['before_1784', 'null'] == 0 and c.loc['before_1784', 'rings'] > 0
assert c.loc['state_cessions', 'points'] == len(sc) and c.loc['royce_cessions', 'rings'] > 0 and c.loc['trail_of_tears', 'lines'] > 0
assert (chk[['bad_rings', 'bad_lines', 'bad_type']] == 0).all().all(), chk
# T-011: every decoded coordinate inside the shared extent
display(assert_inside(chk, EXTENT))
print('ok')

royce max deviation 0.00020 deg (limit 0.00066)
trail max deviation 0.00006 deg (limit 0.00024)


before_1784 max deviation 0.00020 deg (limit 0.00066)


,file,object,features,null,points,rings,bad_rings,lines,bad_lines,bad_type,x0,y0,x1,y1
0,native_land.topojson,royce_cessions,450,7,0,460,0,0,0,0,-100.000000,24.908333,-74.518903,47.441667
1,native_land.topojson,state_cessions,5,0,5,0,0,0,0,0,-77.398640,42.160251,-75.435888,43.241411
2,native_land.topojson,pre1784_lines,100,0,0,0,0,100,0,0,-88.569879,31.400026,-75.130116,43.240059
3,native_land.topojson,before_1784,18,0,0,175,0,0,0,0,-85.165443,30.708697,-66.979613,47.439413
4,removal_routes.topojson,trail_of_tears,2006,0,0,0,0,3898,0,0,-95.119293,33.777546,-83.381199,37.996928


,file,object,x0,y0,x1,y1
0,native_land.topojson,royce_cessions,-100.000000,24.908333,-74.518903,47.441667
1,native_land.topojson,state_cessions,-77.398640,42.160251,-75.435888,43.241411
2,native_land.topojson,pre1784_lines,-88.569879,31.400026,-75.130116,43.240059
3,native_land.topojson,before_1784,-85.165443,30.708697,-66.979613,47.439413
4,removal_routes.topojson,trail_of_tears,-95.119293,33.777546,-83.381199,37.996928


ok


In [20]:
# Summary counts by decade (all three files)
con.register('sc_v', sc.drop(columns='geometry')); con.register('pl_v', pl.drop(columns='geometry'))
print('native_land.topojson:', {k: len(v['geometries']) for k, v in nl['objects'].items()},
      '| royce polygons', sum(g['type'] is not None for g in G), '| royce rows without polygon', sum(g['type'] is None for g in G))
print('T-021 destinations:', {k: int(v) for k, v in roy[roy.is_destination].groupby(['redated', 'assigned_verified']).size().items()}, '| before_1784:', f'{zone_km2:,.0f} km2,', b84.groupby('uncertainty').area_km2.sum().to_dict())
print('reservations.json:', len(rj), 'reserves,', sum(r['has_coords'] for r in rj), 'with a point')
print('removal_routes.topojson:', len(T), 'segments, all undated')
q("""select decade, sum(royce)::int royce_cessions, sum(state)::int state_cession_points, sum(pre)::int pre1784_segments from (
       select (cession_year::int // 10) * 10 as decade, 1 royce, 0 state, 0 pre from roy_v
       union all select (year::int // 10) * 10, 0, 1, 0 from sc_v
       union all select (year::int // 10) * 10, 0, 0, 1 from pl_v)
     group by 1 order by 1""")

native_land.topojson: {'royce_cessions': 450, 'state_cessions': 5, 'pre1784_lines': 100, 'before_1784': 18} | royce polygons 443 | royce rows without polygon 7
T-021 destinations: {('next_usfs_cession', 'n'): 13, ('next_usfs_cession', 'y'): 22, ('no_later_usfs_cession', 'n'): 1, ('no_later_usfs_cession', 'y'): 1, ('todo_no_assigned_date', 'n'): 5} | before_1784: 764,053 km2, {'high': 49593, 'low': 709312, 'medium': 5153}
reservations.json: 66 reserves, 59 with a point
removal_routes.topojson: 2006 segments, all undated


,decade,royce_cessions,state_cession_points,pre1784_segments
0,1760,0,0,81
1,1770,0,0,19
2,1780,6,5,0
3,1790,36,0,0
4,1800,55,0,0
5,1810,71,0,0
6,1820,43,0,0
7,1830,56,0,0
8,1840,21,0,0
9,1850,90,0,0


## Notes for review
- **All dated Royce rows are kept** (changed 2026-11-23 from ≤ 1840). Areas first ceded after 1840 (much of WI, MN, IA and the trans-Mississippi) are drawn as not yet ceded for the whole 1784–1840 slider.
- Royce rows with no polygon in the USFS shapefile are kept with null geometry (`flag = no_geometry`); see §1.
- **USFS dates look one day early** (§1a): consistent with a time-zone shift when the table was built. Kept as stored; a fix (+1 day on the 12:00 UTC dates) needs Lara's decision and a historian check against Royce's schedule.
- Ring winding is as read from the shapefile; the encoder does not re-orient rings.
- **Clipped to the shared extent** (T-011, §1b): cessions wholly west of −100° or north of 47.44° are not in the file; cut ones are flagged `clipped_to_extent`.
- **Removal destinations (T-021, §1c):** re-dated from `destinations_R10.csv`; 19 of its 42 rows are unverified (`assigned_verified = n`). Miami 329/330 move to 1840-11-27, which R10 says may be the treaty that sited the tract, not a cession; Winnebago 267 moves to 1837. Rows with no later USFS date have a null `cession_year` (`year_flag = no_later_usfs_cession`); the web filter must treat null as never ceded.
- **`before_1784` (T-022, §3b)** is built from the R11 lines. Its uncertainty is about where the bounding line runs, not about the interior. Three closures are assumed and flagged (`closure_assumed`): due north from Wood Creek (northern New York, may be too large), due south from Santa Savilla, and augusta_1773 seq 4 (no source anchor; with `USE_A73_SEQ4 = False` the zone closes down the Savannah from Towatuhe to the Little River mouth instead and leaves the 1773 Ceded Lands out).
- **Page note:** Cessions of 1775–83 (Henderson, DeWitt's Corner, Long Island of Holston, Augusta 1783) are not yet drawn.
- **Rivers:** NE 10m courses are modern (dams, channel shifts); the treaties say "south side" of the Ohio and Allegheny, the NE line is the centre line (within the 3 km floor).
